In [1]:
import os
import json
import mne
import numpy as np
import pandas as pd
from scipy.signal import resample
import warnings
warnings.filterwarnings("ignore")

In [2]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [3]:
# root dir
root = 'ADFTD-PS/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,Gender,Age,Group,MMSE
0,sub-001,F,57,A,16
1,sub-002,F,78,A,22
2,sub-003,M,70,A,14
3,sub-004,F,67,A,20
4,sub-005,M,70,A,22
...,...,...,...,...,...
83,sub-084,F,71,F,24
84,sub-085,M,64,F,26
85,sub-086,M,49,F,26
86,sub-087,M,73,F,24


In [4]:
# label mapping: diagnosis code -> label id
label_map = {'A': 1, 'F': 2, 'C': 0}

# build subject info: "sub-XXX" -> (label, pid)
sub_info = {}  # key: subject name, value: (label, pid)
for row in participants.itertuples(index=False):
    sub_name = row[0]          # e.g. 'sub-001'
    diag_code = row[3]         # e.g. 'A' / 'F' / 'C'
    pid = int(sub_name[-3:])   # convert '001' -> 1
    label = label_map[diag_code]
    sub_info[sub_name] = (label, pid)

len(sub_info), list(sub_info.items())[:5]

(88,
 [('sub-001', (1, 1)),
  ('sub-002', (1, 2)),
  ('sub-003', (1, 3)),
  ('sub-004', (1, 4)),
  ('sub-005', (1, 5))])

## Features

In [5]:
from collections import defaultdict

# derivatives root
derivatives_root = os.path.join(root, 'derivatives/eeglab/')
derivatives_root

'ADFTD-PS/derivatives/eeglab/'

In [6]:
# Test for bad channels, sampling freq and shape
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(derivatives_root):
    if 'sub-' in sub:
        sub_path = os.path.join(derivatives_root, sub, 'eeg/')
        # print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                # get bad channels
                bad_channel = raw.info['bads']
                bad_channel_list.append(bad_channel)
                # get sampling frequency
                sampling_freq = raw.info['sfreq']
                sampling_freq_list.append(sampling_freq)
                # get eeg data
                data = raw.get_data()
                data_shape = data.shape
                data_shape_list.append(data_shape)

In [7]:
from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
(19, 151675)
Channel number counter: Counter({19: 88})
Sampling rate counter: Counter({500.0: 88})


In [8]:
# Initialize common channel list
common_channels = []

# Loop through subject folders
for sub in os.listdir(derivatives_root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        for file in os.listdir(sub_path):
            # Change: look for BrainVision .vhdr files
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                current_channels = set(raw.info['ch_names'])

                if not common_channels:
                    common_channels = current_channels
                else:
                    common_channels &= current_channels  # Intersection

# Convert set to list
common_channels = list(common_channels)
# Print result
print(common_channels)
print("Common channels number: ", len(common_channels))

['T4', 'F3', 'O1', 'F7', 'Fp1', 'C4', 'Cz', 'T3', 'Fz', 'Pz', 'F4', 'O2', 'P4', 'P3', 'T6', 'F8', 'Fp2', 'C3', 'T5']
Common channels number:  19


In [10]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [12]:
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
n_channels = len(common_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP)

for sub in sorted(os.listdir(derivatives_root)):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(derivatives_root, sub, 'eeg/')
        print(sub_path)

        print(f"[PASS 1] Subject: {sub}")

        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                print("  reading header:", file_path)
                # only load metadata
                raw = mne.io.read_raw_eeglab(file_path, preload=False)
                # ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']
                # print(raw.ch_names)  
                original_fs = raw.info['sfreq']
                T_raw = raw.n_times
        
                for fs in SAMPLE_RATE_LIST:
                    # resampled length if each trial
                    T_res = int(T_raw * fs / original_fs)
                    # compute number of segments
                    n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                    subject_segment_counts[sub][fs] += n_seg
                    N_total += n_seg
                    print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
        print("-------------------------------------\n")

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200
ADFTD-PS/derivatives/eeglab/sub-001\eeg/
[PASS 1] Subject: sub-001
  reading header: ADFTD-PS/derivatives/eeglab/sub-001\eeg/sub-001_task-photomark_eeg.set
fs=200Hz: T_res=60670, STEP=200, n_seg=302
fs=100Hz: T_res=30335, STEP=200, n_seg=150
fs=50Hz: T_res=15167, STEP=200, n_seg=74
-------------------------------------

ADFTD-PS/derivatives/eeglab/sub-002\eeg/
[PASS 1] Subject: sub-002
  reading header: ADFTD-PS/derivatives/eeglab/sub-002\eeg/sub-002_task-photomark_eeg.set
fs=200Hz: T_res=77402, STEP=200, n_seg=386
fs=100Hz: T_res=38701, STEP=200, n_seg=192
fs=50Hz: T_res=19350, STEP=200, n_seg=95
-------------------------------------

ADFTD-PS/derivatives/eeglab/sub-003\eeg/
[PASS 1] Subject: sub-003
  reading header: ADFTD-PS/derivatives/eeglab/sub-003\eeg/sub-003_task-photomark_eeg.set
fs=200Hz: T_res=54837, STEP=200, n_seg=273
fs=100Hz: T_res=27418, STEP=200, n_seg=136
fs=50Hz: T_res=13709, STEP=200, n_seg=67
----------------------------------

In [13]:
output_root = os.path.join('Processed', sub_folder_path, 'ADFTD-PS')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\ADFTD-PS\X.dat
y path: Processed\L400\ADFTD-PS\y.dat


## PASS 2: Process and store segments into memmap

In [14]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
for sub in sorted(os.listdir(derivatives_root)):
    if 'sub-' in sub:
        label, pid = sub_info[sub]
        sub_path = os.path.join(derivatives_root, sub, 'eeg')
        print(sub_path)
        if sub not in sub_info:
            continue
    
        total_seg_sub = sum(subject_segment_counts[sub][fs] for fs in SAMPLE_RATE_LIST)
        if total_seg_sub == 0:
            continue
        print(f"[PASS 2] Subject: {sub}, expected total segments={total_seg_sub}")
    
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                print("  load:", file_path)
        
                # load full data
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                original_fs = raw.info['sfreq']
                data_raw = raw.get_data().T.astype('float32')  # (C, T) -> (T, C)
                total_seconds_all += data_raw.shape[0] / original_fs
                print("raw shape:", data_raw.shape)
        
                for fs in SAMPLE_RATE_LIST:
                    # resample to target fs
                    data = resample_time_series(data_raw, original_fs, fs)
                    T_res, _ = data.shape
                    print(f"fs={fs}Hz: resampled shape={data.shape}")
        
                    # overlapped sliding window with fixed STEP (in timestamps)
                    starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                    print(f"fs={fs}Hz: segments={len(starts)}")
        
                    for s in starts:
                        if cur >= N_total:
                            raise RuntimeError("Exceeded predicted N_total.")
        
                        X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                        y_mm[cur, 0] = float(label)       # label
                        y_mm[cur, 1] = float(pid)         # subject ID
                        y_mm[cur, 2] = float(fs)          # sample rate (scale)
                        cur += 1
    print("-------------------------------------\n")

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

-------------------------------------

ADFTD-PS/derivatives/eeglab/sub-001\eeg
[PASS 2] Subject: sub-001, expected total segments=526
  load: ADFTD-PS/derivatives/eeglab/sub-001\eeg\sub-001_task-photomark_eeg.set
raw shape: (151675, 19)
fs=200Hz: resampled shape=(60670, 19)
fs=200Hz: segments=302
fs=100Hz: resampled shape=(30335, 19)
fs=100Hz: segments=150
fs=50Hz: resampled shape=(15167, 19)
fs=50Hz: segments=74
-------------------------------------

ADFTD-PS/derivatives/eeglab/sub-002\eeg
[PASS 2] Subject: sub-002, expected total segments=673
  load: ADFTD-PS/derivatives/eeglab/sub-002\eeg\sub-002_task-photomark_eeg.set
raw shape: (193505, 19)
fs=200Hz: resampled shape=(77402, 19)
fs=200Hz: segments=386
fs=100Hz: resampled shape=(38701, 19)
fs=100Hz: segments=192
fs=50Hz: resampled shape=(19350, 19)
fs=50Hz: segments=95
-------------------------------------

ADFTD-PS/derivatives/eeglab/sub-003\eeg
[PASS 2] Subject: sub-003, expected total segments=476
  load: ADFTD-PS/derivatives/eeg

## Load and check the processed data

In [15]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 45258
  T = 400
  C = 19
  X_path = Processed\L400\ADFTD-PS\X.dat
  y_path = Processed\L400\ADFTD-PS\y.dat
-------------------------------------
Subject ID 001: X shape=(526, 400, 19), y shape=(526, 3)
Subject ID 002: X shape=(673, 400, 19), y shape=(673, 3)
Subject ID 003: X shape=(476, 400, 19), y shape=(476, 3)
Subject ID 004: X shape=(582, 400, 19), y shape=(582, 3)
Subject ID 005: X shape=(508, 400, 19), y shape=(508, 3)
Subject ID 006: X shape=(677, 400, 19), y shape=(677, 3)
Subject ID 007: X shape=(742, 400, 19), y shape=(742, 3)
Subject ID 008: X shape=(663, 400, 19), y shape=(663, 3)
Subject ID 009: X shape=(712, 400, 19), y shape=(712, 3)
Subject ID 010: X shape=(516, 400, 19), y shape=(516, 3)
Subject ID 011: X shape=(505, 400, 19), y shape=(505, 3)
Subject ID 012: X shape=(585, 400, 19), y shape=(585, 3)
Subject ID 013: X shape=(802, 400, 19), y shape=(802, 3)
Subject ID 014: X shape=(865, 400, 19), y shape=(865, 3)
Subject ID 015: X shape=(494, 400, 19), y sha